# Cruce H×M×interactividad — LocalScore 3783 (CACIC)

Atajo quirúrgico para el short paper CACIC 2026.

- **Input:** `data/processed/localscore_runs_enriched.csv` (3.783 corridas del cache JSON).
- **Reglas:** mismas que notebooks 2 y 4 (`label_h`, bins M1–M5, umbrales 10/25 tok/s).
- **Salidas (sin pisar JAIIO):**
  - `data/processed/localscore_runs_labeled_3783.csv`
  - `data/processed/fig03_hardware_model_perf_table_3783.csv`
  - `private/cacic-2026/figures/fig03_hardware_model_perf_percent_clean.{png,svg,pdf}`
  - `private/cacic-2026/numeros_clave_3783.md`

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BASE = Path("..")
DATA_DIR = BASE / "data" / "processed"
FIG_DIR = BASE / "private" / "cacic-2026" / "figures"
CACIC_DIR = BASE / "private" / "cacic-2026"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_PATH = DATA_DIR / "localscore_runs_enriched.csv"
OUT_LABELED = DATA_DIR / "localscore_runs_labeled_3783.csv"
OUT_TABLE = DATA_DIR / "fig03_hardware_model_perf_table_3783.csv"
OUT_NUMS = CACIC_DIR / "numeros_clave_3783.md"

OUT_FIG_PCT_CLEAN = FIG_DIR / "fig03_hardware_model_perf_percent_clean.png"
OUT_FIG_PCT_CLEAN_SVG = FIG_DIR / "fig03_hardware_model_perf_percent_clean.svg"
OUT_FIG_PCT_CLEAN_PDF = FIG_DIR / "fig03_hardware_model_perf_percent_clean.pdf"

H_ORDER = ["H1", "H2", "H3", "H4", "H5", "H6", "H7"]
M_ORDER = ["M1", "M2", "M3", "M4", "M5"]
PERF_ORDER = ["Batch", "Interactivo mínimo", "Interactivo fluido"]

MIN_TPS = 10.0
FLUENT_TPS = 25.0
LOW_N_THRESHOLD = 10

print("Input:", ENRICHED_PATH)
print("Figuras →", FIG_DIR)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Perfiles H0-H7 (copiado de caracterizacion_hardware_perfiles_localscore.ipynb)
CPU_H0_MAX_EXCL = 4
CPU_H1_MAX_EXCL = 8
CPU_H2_MAX_EXCL = 16

GPU_H4_MAX_EXCL = 8
GPU_H5_MIN = 8
GPU_H5_MAX_EXCL = 24
GPU_H6_MIN = 24

DATACENTER_KEYWORDS = [
    "H100", "H200", "A100", "A40", "L40S",
    "MI250", "MI300", "MI300X",
    "TESLA", "V100", "P100", "A800",
    "GAUDI", "GAUD",
    "SERVER EDITION", "BLACKWELL SERVER",
]

def is_datacenter_gpu(accelerator_name: str) -> bool:
    if accelerator_name is None or pd.isna(accelerator_name):
        return False
    name = str(accelerator_name).upper()
    return any(kw in name for kw in DATACENTER_KEYWORDS)


def label_h(row: pd.Series) -> str:
    atype = row.get("accelerator_type", None)
    atype = str(atype).upper() if atype is not None and not pd.isna(atype) else None

    if atype is None:
        return "UNKNOWN"

    if atype == "CPU":
        ram = row.get("ram_gb", None)
        if ram is None or pd.isna(ram):
            return "UNKNOWN"
        ram = float(ram)
        if ram < CPU_H0_MAX_EXCL:
            return "H0_CPU_EDGE"
        if ram < CPU_H1_MAX_EXCL:
            return "H1_CPU_LOW"
        if ram < CPU_H2_MAX_EXCL:
            return "H2_CPU_MED"
        return "H3_CPU_HIGH"

    if atype == "GPU":
        name = row.get("accelerator_name", None)
        vram = row.get("memory_gb", None)

        if is_datacenter_gpu(name):
            return "H7_SERVER"

        if vram is None or pd.isna(vram):
            return "UNKNOWN"
        vram = float(vram)

        if vram >= GPU_H6_MIN:
            return "H6_GPU_HIGH"
        if vram >= GPU_H5_MIN:
            return "H5_GPU_MED"
        return "H4_GPU_LOW"

    return "UNKNOWN"

print("label_h listo")

In [ ]:
runs = pd.read_csv(ENRICHED_PATH)
need = {"accelerator_type", "ram_gb", "memory_gb", "accelerator_name", "params", "avg_gen_tps"}
missing = need - set(runs.columns)
if missing:
    raise ValueError(f"Faltan columnas en enriched: {sorted(missing)}")

runs["profile_h"] = runs.apply(label_h, axis=1)
unknown_rate = (runs["profile_h"] == "UNKNOWN").mean() * 100.0
print(f"Corridas: {len(runs)}")
print(f"unknown_rate: {unknown_rate:.2f}%")
print(runs["profile_h"].value_counts(dropna=False).sort_index())
if unknown_rate >= 1.0:
    print("ADVERTENCIA: unknown_rate >= 1% — revisar mapeo de columnas.")

In [ ]:
def short_h(profile_h: str) -> str:
    if pd.isna(profile_h):
        return "UNKNOWN"
    p = str(profile_h)
    if p.startswith("H") and "_" in p:
        return p.split("_", 1)[0]
    if p in H_ORDER or p == "H0":
        return p
    return "UNKNOWN"


def profile_m(params):
    if pd.isna(params):
        return "UNKNOWN"
    p = float(params)
    if p < 1e9:
        return "M1"
    if p < 4e9:
        return "M2"
    if p < 8e9:
        return "M3"
    if p < 16e9:
        return "M4"
    return "M5"


def perf_category(avg_gen_tps):
    x = pd.to_numeric(avg_gen_tps, errors="coerce")
    if pd.isna(x):
        return np.nan
    if x < MIN_TPS:
        return "Batch"
    if x < FLUENT_TPS:
        return "Interactivo mínimo"
    return "Interactivo fluido"

runs["H"] = runs["profile_h"].apply(short_h)
runs["M"] = runs["params"].apply(profile_m)
runs["perf_category"] = runs["avg_gen_tps"].apply(perf_category)

plot_runs = runs[
    runs["H"].isin(H_ORDER)
    & runs["M"].isin(M_ORDER)
    & runs["perf_category"].isin(PERF_ORDER)
].copy()

print("Corridas totales:", len(runs))
print("Corridas en figura (H1–H7):", len(plot_runs))
print("Cobertura H:\n", plot_runs["H"].value_counts().reindex(H_ORDER, fill_value=0))
print("Cobertura M:\n", plot_runs["M"].value_counts().reindex(M_ORDER, fill_value=0))

In [ ]:
counts = (
    plot_runs.groupby(["H", "M", "perf_category"], observed=True)
    .size()
    .rename("n")
    .reset_index()
)

wide_abs = (
    counts.pivot_table(index=["H", "M"], columns="perf_category", values="n", fill_value=0)
    .reindex(columns=PERF_ORDER, fill_value=0)
)
wide_pct = wide_abs.div(wide_abs.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100.0

out_table = counts.copy()
cell_totals = plot_runs.groupby(["H", "M"]).size().rename("n_cell")
out_table = out_table.merge(cell_totals.reset_index(), on=["H", "M"], how="left")
out_table["pct"] = (out_table["n"] / out_table["n_cell"] * 100).round(1)
out_table.to_csv(OUT_TABLE, index=False)

runs.to_csv(OUT_LABELED, index=False)
print("Tabla:", OUT_TABLE)
print("Labeled:", OUT_LABELED)
out_table.head(10)

In [ ]:
# Variante clean (copiada de cruce_hardware_modelos_grafico3.ipynb)
PERF_COLORS_CLEAN = {
    "Batch": "#E69F00",
    "Interactivo mínimo": "#56B4E9",
    "Interactivo fluido": "#009E73",
}

plot_pct = wide_pct.reindex(
    index=pd.MultiIndex.from_product([H_ORDER, M_ORDER], names=["H", "M"]),
    columns=PERF_ORDER,
).fillna(0)
plot_abs = wide_abs.reindex(
    index=pd.MultiIndex.from_product([H_ORDER, M_ORDER], names=["H", "M"]),
    columns=PERF_ORDER,
).fillna(0)
cell_n = plot_abs.sum(axis=1)

n_m = len(M_ORDER)
bar_w = 0.195
group_gap = 0.24
block_w = n_m * bar_w

x_positions = {}
xticks = []
xticklabels = []
h_centers = []

for i, h in enumerate(H_ORDER):
    block_start = i * (block_w + group_gap)
    h_centers.append(block_start + (block_w - bar_w) / 2)
    for j, m in enumerate(M_ORDER):
        x = block_start + j * bar_w
        x_positions[(h, m)] = x
        xticks.append(x)
        xticklabels.append(m)

fig, ax = plt.subplots(figsize=(12.0, 4.25), dpi=280)

for h in H_ORDER:
    for m in M_ORDER:
        x = x_positions[(h, m)]
        bottom = 0.0
        for cat in PERF_ORDER:
            y = float(plot_pct.loc[(h, m), cat])
            if y > 0:
                ax.bar(
                    x,
                    y,
                    width=bar_w * 0.94,
                    bottom=bottom,
                    color=PERF_COLORS_CLEAN[cat],
                    edgecolor="white",
                    linewidth=0.50,
                )
            bottom += y

        n = float(cell_n.loc[(h, m)])
        if n > 0 and n < LOW_N_THRESHOLD:
            ax.text(x, 100.9, "*", ha="center", va="bottom", fontsize=9, color="#3a3a3a", fontweight="bold")

for i in range(1, len(H_ORDER)):
    prev_end = (i - 1) * (block_w + group_gap) + (n_m - 1) * bar_w + bar_w / 2
    next_start = i * (block_w + group_gap) - bar_w / 2
    sep_x = (prev_end + next_start) / 2
    ax.axvline(sep_x, color="#d0d0d0", lw=0.75, zorder=0)

ax.set_ylim(0, 101.5)
ax.set_ylabel("Porcentaje de corridas (%)", fontsize=10)
ax.set_xlabel("Perfil de modelo (M)", fontsize=10)
ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels, fontsize=8)
ax.tick_params(axis="y", labelsize=9)
ax.grid(axis="y", alpha=0.22, linestyle="-", linewidth=0.55)

for xc, h in zip(h_centers, H_ORDER):
    ax.text(xc, -0.17, h, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=9)

perf_handles = [Patch(facecolor=PERF_COLORS_CLEAN[c], edgecolor="none", label=c) for c in PERF_ORDER]
ax.legend(
    handles=perf_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3,
    frameon=False,
    fontsize=9,
    handlelength=1.5,
    columnspacing=1.0,
)

fig.tight_layout(rect=[0, 0.065, 1, 1])
fig.savefig(OUT_FIG_PCT_CLEAN, bbox_inches="tight", pad_inches=0.02)
fig.savefig(OUT_FIG_PCT_CLEAN_SVG, bbox_inches="tight", pad_inches=0.02)
fig.savefig(OUT_FIG_PCT_CLEAN_PDF, bbox_inches="tight", pad_inches=0.02)
plt.show()
plt.close(fig)

print("PNG:", OUT_FIG_PCT_CLEAN)
print("SVG:", OUT_FIG_PCT_CLEAN_SVG)
print("PDF:", OUT_FIG_PCT_CLEAN_PDF)

In [ ]:
def cell_stats(h, m):
    sub = plot_runs[(plot_runs["H"] == h) & (plot_runs["M"] == m)]
    n = len(sub)
    if n == 0:
        return {"H": h, "M": m, "n": 0}
    vc = sub["perf_category"].value_counts()
    batch = int(vc.get("Batch", 0))
    minimo = int(vc.get("Interactivo mínimo", 0))
    fluido = int(vc.get("Interactivo fluido", 0))
    inter = minimo + fluido
    return {
        "H": h,
        "M": m,
        "n": n,
        "batch_n": batch,
        "batch_pct": round(100 * batch / n, 1),
        "min_n": minimo,
        "min_pct": round(100 * minimo / n, 1),
        "fluido_n": fluido,
        "fluido_pct": round(100 * fluido / n, 1),
        "interactivo_pct": round(100 * inter / n, 1),
    }

key_cells = [("H3", "M2"), ("H3", "M4"), ("H5", "M2"), ("H5", "M4"), ("H6", "M2"), ("H6", "M4")]
stats = [cell_stats(h, m) for h, m in key_cells]

n_total = len(runs)
n_accels = runs["accelerator_id"].nunique()
n_systems = runs["system_id"].nunique()
n_combos = runs.groupby(["system_id", "accelerator_id"]).ngroups
n_cpus = runs["cpu_name"].nunique()
n_models = runs["model_name"].nunique()

lines = []
lines.append("# Números clave — LocalScore 3783 (CACIC)\n")
lines.append("Generado por `notebooks/cruce_hardware_modelos_3783.ipynb`.\n")
lines.append("## Diversidad\n")
lines.append(f"- Corridas: **{n_total}**")
lines.append(f"- Configuraciones únicas (system×acelerador): **{n_combos}**")
lines.append(f"- Aceleradores distintos: **{n_accels}**")
lines.append(f"- Sistemas distintos: **{n_systems}**")
lines.append(f"- CPUs distintas (nombre): **{n_cpus}**")
lines.append(f"- Modelos distintos: **{n_models}**")
lines.append(f"- unknown_rate (profile_h): **{unknown_rate:.2f}%**\n")
lines.append("## Celdas clave H×M (para abstract / §4)\n")
lines.append("| Celda | n | Batch % | Interactivo mín. % | Fluido % | Interactivo total % |")
lines.append("|---|---:|---:|---:|---:|---:|")
for s in stats:
    lines.append(
        f"| {s['H']}×{s['M']} | {s['n']} | {s.get('batch_pct', '—')} | "
        f"{s.get('min_pct', '—')} | {s.get('fluido_pct', '—')} | {s.get('interactivo_pct', '—')} |"
    )
lines.append("\n## Frases listas para pegar\n")
h3m4 = next(s for s in stats if s["H"] == "H3" and s["M"] == "M4")
h5m4 = next(s for s in stats if s["H"] == "H5" and s["M"] == "M4")
h6m4 = next(s for s in stats if s["H"] == "H6" and s["M"] == "M4")
h3m2 = next(s for s in stats if s["H"] == "H3" and s["M"] == "M2")
lines.append(
    f"- Se analizan **{n_total}** ejecuciones del benchmark LocalScore sobre "
    f"**~{n_combos}** configuraciones únicas de hardware (**{n_accels}** aceleradores distintos)."
)
lines.append(
    f"- Quiebre M4: H3×M4 es **{h3m4['batch_pct']}%** batch ({h3m4['batch_n']}/{h3m4['n']}); "
    f"H5×M4 **{h5m4['fluido_pct']}%** fluido ({h5m4['fluido_n']}/{h5m4['n']}); "
    f"H6×M4 **{h6m4['fluido_pct']}%** fluido ({h6m4['fluido_n']}/{h6m4['n']})."
)
lines.append(
    f"- Robustez M2: H3×M2 alcanza **{h3m2['interactivo_pct']}%** interactivo "
    f"({h3m2['interactivo_pct'] and ''}{h3m2.get('fluido_pct')}% fluido; n={h3m2['n']})."
)

OUT_NUMS.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(OUT_NUMS.read_text())
print("Escrito:", OUT_NUMS)